# Yatharth Music AI — Free GPU real-AI test v2

This notebook starts ACE-Step 1.5, waits for a real health response, starts Yatharth in real-AI mode, verifies engine reachability, and then creates a temporary HTTPS link.

Free Colab GPU/runtime availability is not guaranteed. This is for validation, not permanent hosting.

In [ ]:
!nvidia-smi || true
!rm -rf /content/ACE-Step-1.5 /content/yatharth-music-ai
!git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git /content/ACE-Step-1.5
!git clone --depth 1 https://github.com/rampaulsaini/yatharth-music-ai.git /content/yatharth-music-ai
%cd /content/ACE-Step-1.5
!pip -q install uv
!uv sync --frozen
%cd /content/yatharth-music-ai
!pip -q install -r requirements.txt

In [ ]:
# Start ACE-Step and wait up to 3 minutes for its health endpoint.
import subprocess, time, os, requests
log = open('/content/acestep.log', 'w')
proc = subprocess.Popen(['uv','run','python','-m','acestep.api_server','--host','127.0.0.1','--port','8001'], stdout=log, stderr=subprocess.STDOUT, cwd='/content/ACE-Step-1.5')
ready = False
last_error = None
for _ in range(90):
    time.sleep(2)
    if proc.poll() is not None:
        break
    try:
        r = requests.get('http://127.0.0.1:8001/health', timeout=5)
        print('ACE-Step:', r.status_code, r.text[:500])
        if r.status_code < 500:
            ready = True
            break
    except Exception as e:
        last_error = str(e)
print('ACE-Step READY:', ready, 'PID:', proc.pid, 'exit:', proc.poll())
if not ready:
    print('Last connection error:', last_error)
    print(open('/content/acestep.log', errors='ignore').read()[-8000:])
    raise RuntimeError('ACE-Step did not become ready.')

In [ ]:
# Start Yatharth and require engine_reachable=true before continuing.
import subprocess, time, requests, os
ylog = open('/content/yatharth.log', 'w')
env = os.environ.copy()
env['DEMO_MODE'] = 'false'
env['MUSIC_ENGINE_URL'] = 'http://127.0.0.1:8001'
env['TRUST_PROXY'] = 'false'
backend = subprocess.Popen(['python','-m','uvicorn','main:app','--host','0.0.0.0','--port','8000','--proxy-headers','--forwarded-allow-ips','127.0.0.1'], stdout=ylog, stderr=subprocess.STDOUT, cwd='/content/yatharth-music-ai', env=env)
ready = False
for _ in range(20):
    time.sleep(2)
    try:
        r = requests.get('http://127.0.0.1:8000/api/health', timeout=10)
        body = r.json()
        print(body)
        if r.status_code == 200 and body.get('ok') and body.get('engine_reachable'):
            ready = True
            break
    except Exception as e:
        print('Waiting for Yatharth:', e)
print('Yatharth READY:', ready, 'PID:', backend.pid)
if not ready:
    print(open('/content/yatharth.log', errors='ignore').read()[-8000:])
    raise RuntimeError('Yatharth is not ready or cannot reach ACE-Step.')

In [ ]:
# Create a temporary Cloudflare HTTPS URL only after both services are ready.
import subprocess, re, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
tunnel_log = open('/content/cloudflared.log', 'w')
tunnel = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(30):
    time.sleep(2)
    text = open('/content/cloudflared.log', errors='ignore').read()
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if match:
        public_url = match.group(0)
        break
print('YATHARTH PUBLIC LINK:', public_url)
if not public_url:
    print(open('/content/cloudflared.log', errors='ignore').read()[-5000:])
    raise RuntimeError('Cloudflare public URL was not created.')
print('Keep this Colab runtime running while testing from the phone.')

## Real AI test

Open the printed **YATHARTH PUBLIC LINK** on your phone. Generate a **10–30 second** song first.

Success means: phone → Yatharth UI → FastAPI → ACE-Step → actual audio file. A demo test tone does not count.

In [ ]:
print('--- ACE-Step log ---')
print(open('/content/acestep.log', errors='ignore').read()[-8000:])
print('--- Yatharth log ---')
print(open('/content/yatharth.log', errors='ignore').read()[-8000:])
print('--- Cloudflare log ---')
print(open('/content/cloudflared.log', errors='ignore').read()[-5000:])